# 03. Обучение RL-агента и бэктестинг

В этом ноутбуке переходим к обучению RL-агента.


In [1]:
from pathlib import Path
import json
import pickle
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = str((repo_root / "src").resolve())
config_path = repo_root / "config" / "basis_strategy_config.json"

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from crypto_rl_bot.config import load_config
from crypto_rl_bot.features import get_feature_columns, clean_data_for_features
from crypto_rl_bot.train import (
    split_timeseries_data,
    train_qlearning_agent,
    evaluate_qlearning_agent,
)
from crypto_rl_bot.qlearning import QLearningAgent

print("Корень проекта:", repo_root)
print("Путь к конфигу:", config_path)

Корень проекта: /Users/coyc_kari/Final_project_MLfr_team5
Путь к конфигу: /Users/coyc_kari/Final_project_MLfr_team5/config/basis_strategy_config.json


In [2]:
cfg = load_config(config_path)

print("Спот:", cfg.data.spot_symbol)
print("Фьючерс:", cfg.data.futures_symbol)
print("Интервал:", cfg.data.interval)

print("\nПараметры RL:")
print("episodes:", cfg.rl.episodes)
print("alpha:", cfg.rl.alpha)
print("gamma:", cfg.rl.gamma)
print("epsilon_start:", cfg.rl.epsilon_start)
print("epsilon_end:", cfg.rl.epsilon_end)
print("epsilon_decay:", cfg.rl.epsilon_decay)

print("\nПараметры baseline:")
print("initial_capital:", cfg.baseline.initial_capital)
print("transaction_cost:", cfg.baseline.transaction_cost)
print("enter_zscore:", cfg.baseline.enter_zscore)
print("exit_zscore:", cfg.baseline.exit_zscore)

Спот: BTC-USDT
Фьючерс: XBTUSDTM
Интервал: 1day

Параметры RL:
episodes: 300
alpha: 0.1
gamma: 0.95
epsilon_start: 0.2
epsilon_end: 0.02
epsilon_decay: 0.97

Параметры baseline:
initial_capital: 10000.0
transaction_cost: 0.001
enter_zscore: 1.5
exit_zscore: 0.3


## 2. Загрузка подготовленного датасета

Загрузим датасет с признаками, который был подготовлен на предыдущем шаге.  
Для RL важно работать с временным рядом без перемешивания, поэтому дальше мы будем делить выборку строго по времени

In [3]:
cfg = load_config(config_path)

processed_path = repo_root / "data" / "processed" / "rl_feature_dataset.parquet"
df = pd.read_parquet(processed_path)

feature_cols = get_feature_columns()
df = clean_data_for_features(df, feature_cols)

print("Размер датасета:", df.shape)
print("Количество признаков:", len(feature_cols))

df.head()

Размер датасета: (730, 40)
Количество признаков: 17


,ts,spot_open,spot_high,spot_low,spot_close,spot_volume,fut_open,fut_high,fut_low,fut_close,...,above_sma_50,rsi,ema_12,ema_26,macd,macd_signal,macd_hist,momentum_10,rolling_corr,log_spread
0,2024-03-30 00:00:00+00:00,69846.2,70331.0,69528.1,69570.9,1012.860448,69864.6,70400.0,69566.3,69610.7,...,0,51.106565,69570.900000,69570.900000,0.000000,0.000000,0.000000,0.255927,0.999955,0.000572
1,2024-03-31 00:00:00+00:00,69578.4,71349.0,69553.4,71264.5,1171.486226,69615.7,71478.5,69603.5,71379.7,...,0,51.106565,69831.453846,69696.351852,135.101994,27.020399,108.081595,0.255927,0.999955,0.001615
2,2024-04-01 00:00:00+00:00,71257.7,71274.8,68039.1,69653.0,3362.179645,71369.9,71369.9,68109.2,69662.3,...,0,51.106565,69803.999408,69693.140604,110.858805,43.788080,67.070725,0.255927,0.999955,0.000134
3,2024-04-02 00:00:00+00:00,69653.0,69677.5,64597.5,65482.9,5617.674202,69654.7,69682.0,64579.9,65475.8,...,0,51.106565,69139.214884,69381.270929,-242.056045,-13.380745,-228.675300,0.255927,0.999955,-0.000108
4,2024-04-03 00:00:00+00:00,65473.1,66890.9,64455.7,65959.2,3870.986185,65474.8,66952.6,64474.2,65955.9,...,0,51.106565,68649.981825,69127.784194,-477.802369,-106.265070,-371.537299,0.255927,0.999955,-0.000050


## Разделим данные на train и test


In [4]:
train_ratio = 0.7
split_idx = int(len(df) * train_ratio)

df_train = df.iloc[:split_idx].reset_index(drop=True)
df_test = df.iloc[split_idx:].reset_index(drop=True)

print("Обучающая выборка:", df_train.shape)
print("Тестовая выборка:", df_test.shape)
print("Диапазон Train:", df_train["ts"].min(), "->", df_train["ts"].max())
print("Диапазон Test:", df_test["ts"].min(), "->", df_test["ts"].max())

Обучающая выборка: (510, 40)
Тестовая выборка: (220, 40)
Диапазон Train: 2024-03-30 00:00:00+00:00 -> 2025-08-21 00:00:00+00:00
Диапазон Test: 2025-08-22 00:00:00+00:00 -> 2026-03-29 00:00:00+00:00


In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(df_train[feature_cols])
X_test = scaler.transform(df_test[feature_cols])

train_data = {
    "df": df_train,
    "features": X_train,
}

test_data = {
    "df": df_test,
    "features": X_test,
}

state_dim = X_train.shape[1] + 1  
print("State dim:", state_dim)

State dim: 18


## State, action и reward

- `0` — neutral, без позиции
- `1` — long spot + short futures
- `2` — short spot + long futures



##  Обучение Q-learning агента

In [ ]:
agent, history_df = train_qlearning_agent(
    train_data=train_data,
    state_dim=state_dim,
    episodes=cfg.rl.episodes,
    alpha=cfg.rl.alpha,
    gamma=cfg.rl.gamma,
    epsilon_start=cfg.rl.epsilon_start,
    epsilon_end=cfg.rl.epsilon_end,
    epsilon_decay=cfg.rl.epsilon_decay,
    initial_capital=cfg.baseline.initial_capital,
    transaction_cost=cfg.baseline.transaction_cost,
    position_size_ratio=0.5,
    n_bins=3,
    random_state=42,
)

history_df.head()

## 7. Динамика обучения

Посмотрим, как менялись суммарная награда и итоговая стоимость портфеля по эпизодам.  
Если обучение идет стабильно, мы должны увидеть, что поведение агента со временем становится более устойчивым.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_df["episode"], history_df["total_reward"])
plt.title("Суммарная награда по эпизодам")
plt.xlabel("Эпизод")
plt.ylabel("Суммарная награда")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_df["episode"], history_df["final_portfolio"])
plt.title("Итоговая стоимость портфеля по эпизодам")
plt.xlabel("Эпизод")
plt.ylabel("Стоимость портфеля")
plt.grid(True)
plt.show()

In [ ]:
train_eval = evaluate_qlearning_agent(
    agent=agent,
    data=train_data,
    initial_capital=cfg.baseline.initial_capital,
    transaction_cost=cfg.baseline.transaction_cost,
    position_size_ratio=0.5,
)

test_eval = evaluate_qlearning_agent(
    agent=agent,
    data=test_data,
    initial_capital=cfg.baseline.initial_capital,
    transaction_cost=cfg.baseline.transaction_cost,
    position_size_ratio=0.5,
)

metrics_df = pd.DataFrame(
    [
        {"dataset": "train", **{k: v for k, v in train_eval.items() if not isinstance(v, np.ndarray)}},
        {"dataset": "test", **{k: v for k, v in test_eval.items() if not isinstance(v, np.ndarray)}},
    ]
)

metrics_df